<a href="https://colab.research.google.com/github/MichalSlowakiewicz/Hybrid-Graph-Neural-Networks/blob/master/core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Sequential, Linear, ReLU
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import GCNConv, GATConv, GINConv
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. POBRANIE DANYCH I NEGATIVE SAMPLING
# ==========================================
print("Pobieranie zbioru danych Cora...")
dataset = Planetoid(root='data/Cora', name='Cora')
data = dataset[0]

transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.2,
    is_undirected=True,
    add_negative_train_samples=True
)
train_data, val_data, test_data = transform(data)
print(f"Dane podzielone! Gotowe do treningu.\n")

# ==========================================
# 2. DEFINICJE MODELI (3 POJEDYNCZE + 5 HYBRYD)
# ==========================================
def decode(z, edge_label_index):
    return (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=-1)

# --- MODELE POJEDYNCZE ---
class GCN_Net(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

class GAT_Net(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=1)
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1)
    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

class GIN_Net(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        mlp1 = Sequential(Linear(in_channels, hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINConv(mlp1)
        mlp2 = Sequential(Linear(hidden_channels, hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINConv(mlp2)
    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

# --- MODELE HYBRYDOWE ---
class Hybrid_GCN_GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1)
    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

class Hybrid_GIN_GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        mlp = Sequential(Linear(in_channels, hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINConv(mlp)
        self.conv2 = GATConv(hidden_channels, hidden_channels, heads=1)
    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

class Hybrid_GCN_GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        mlp = Sequential(Linear(hidden_channels, hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINConv(mlp)
    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

class Hybrid_GAT_GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=1)
        mlp = Sequential(Linear(hidden_channels, hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv2 = GINConv(mlp)
    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

class Hybrid_GIN_GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        mlp = Sequential(Linear(in_channels, hidden_channels), ReLU(), Linear(hidden_channels, hidden_channels))
        self.conv1 = GINConv(mlp)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

# ==========================================
# 3. FUNKCJA TRENUJĄCA
# ==========================================
def train_and_test(model, name):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = torch.nn.BCEWithLogitsLoss()

    for epoch in range(1, 41): # 40 epok dla każdego modelu
        model.train()
        optimizer.zero_grad()
        z = model.encode(train_data.x, train_data.edge_index)
        out = decode(z, train_data.edge_label_index)
        loss = criterion(out, train_data.edge_label.float())
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        z = model.encode(test_data.x, test_data.edge_index)
        out = decode(z, test_data.edge_label_index).sigmoid()
        auc = roc_auc_score(test_data.edge_label.cpu().numpy(), out.cpu().numpy())
        print(f"[{name}] AUC: {auc:.4f}")
        return auc



In [ ]:
# ==========================================
# 4. URUCHOMIENIE EKSPERYMENTU
# ==========================================
in_channels = dataset.num_features
hidden_channels = 64

# Słownik do przechowywania wszystkich modeli i ich wyników
models = {
    "GCN": GCN_Net(in_channels, hidden_channels),
    "GAT": GAT_Net(in_channels, hidden_channels),
    "GIN": GIN_Net(in_channels, hidden_channels),
    "GCN-GAT": Hybrid_GCN_GAT(in_channels, hidden_channels),
    "GIN-GAT": Hybrid_GIN_GAT(in_channels, hidden_channels),
    "GCN-GIN": Hybrid_GCN_GIN(in_channels, hidden_channels),
    "GAT-GIN": Hybrid_GAT_GIN(in_channels, hidden_channels),
    "GIN-GCN": Hybrid_GIN_GCN(in_channels, hidden_channels)
}

results = {}
print("Rozpoczynamy masowy trening 8 modeli...\n")
for name, model in models.items():
    results[name] = train_and_test(model, name)



In [ ]:
# ==========================================
# 5. GENEROWANIE WYKRESU (Jak w artykule!)
# ==========================================
names = list(results.keys())
scores = list(results.values())

# Kolorowanie: 3 pierwsze (Pojedyncze) na niebiesko, 5 kolejnych (Hybrydy) na pomarańczowo
colors = ['#1f77b4']*3 + ['#ff7f0e']*5

plt.figure(figsize=(12, 6))
bars = plt.bar(names, scores, color=colors, edgecolor='black', alpha=0.8)

# Dodanie wartości liczbowych nad słupkami
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.005, f"{yval:.3f}",
             ha='center', va='bottom', fontweight='bold')

# Oś Y od 0.5 do 1.0 (bo poniżej 0.5 to model losowy)
plt.ylim(0.5, 1.05)
plt.ylabel('Wynik AUC (Link Prediction)', fontsize=12)
plt.title('Porównanie modeli pojedynczych i hybrydowych GNN', fontsize=14, fontweight='bold')

# Legenda (symulowana)
import matplotlib.patches as mpatches
single_patch = mpatches.Patch(color='#1f77b4', label='Modele Pojedyncze')
hybrid_patch = mpatches.Patch(color='#ff7f0e', label='Modele Hybrydowe')
plt.legend(handles=[single_patch, hybrid_patch], loc='lower right', fontsize=11)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()